# Упражнение: Окна, скейлер по train и прогноз рекуррентной сетью

## Задание

В `input.csv` — месячный ряд из 72 точек (`t, y`) с трендом и годовой сезонностью.
В `weights.json` — веса маленькой рекуррентной сети (Elman RNN, вход 1, скрытое состояние H = 4)
в раскладке PyTorch `nn.RNN`: `W_ih` (4 × 1), `W_hh` (4 × 4), `b_ih` (4), `b_hh` (4).
Рекуррентные веса **заморожены**; обучать будем только линейную «голову» поверх последнего
скрытого состояния (так устроены echo state networks — и это уже настоящий прогноз сетью).

1. **Разбиение по времени:** train — точки t = 0..47, test — t = 48..71.
2. **Нормализация:** `MinMaxScaler`, обученный **только на y train-части** (t < 48), применить ко всему ряду.
3. **Окна:** длина L = 12, горизонт 1. Для каждого t от 12 до 71 вход — масштабированные
   значения t − 12 … t − 1, цель — масштабированное y(t). Окно — обучающее, если t < 48,
   иначе тестовое (тестовые окна могут опираться на историю из train — это прошлое, утечки нет).
4. **Прямой проход RNN** по каждому окну, h₀ = 0, для 12 шагов:
   hₜ = tanh(W_ih·xₜ + b_ih + W_hh·hₜ₋₁ + b_hh); берём последнее состояние h₁₂.
5. **Голова:** `LinearRegression()` на h₁₂ обучающих окон → масштабированная цель.
   Предскажите тестовые окна и верните прогноз в исходный масштаб (`inverse_transform`).
6. **Бейзлайн:** наивный прогноз — последнее значение окна, y(t − 1).

## Подсказки

- Сколько окон получится? Ряд длины 72 и окно 12 дают 60 пар «окно → цель»: 36 обучающих и 24 тестовых.
- Скейлер, обученный на всём ряде, «знает» максимум теста — это утечка. Здесь, наоборот, тест выйдет
  за пределы [0, 1]: из-за тренда значения в конце ряда больше всего, что видел train.
- Окна: `X = np.array([z[i-12:i] for i in range(12, 72)])`; для окна цикл
  `h = np.tanh(W_ih[:, 0] * x + b_ih + W_hh @ h + b_hh)` по 12 значениям; голова — регрессия
  матрицы состояний (60 × 4) на цели, потом `scaler.inverse_transform(pred.reshape(-1, 1))`.

## Ожидаемый формат выходного файла

CSV с колонками `t, y_true, y_naive, y_rnn` — 24 строки для тестовых t = 48..71 по порядку;
значения округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/rnn_lstm_time_series
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.